# Evaluating Agents

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/agent-design-patterns/10-evaluating-agents

We simulate stochastic agents to show why a single run lies, and compute pass@k, consistency, and trajectory efficiency.

Self-contained: NumPy + matplotlib only. No torch, no sklearn, no network, no API keys.

> **To save your work:** click **Copy to Drive** at the top, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Dark style matching the site theme.
plt.style.use('dark_background')
plt.rcParams.update({
    'axes.edgecolor': '#475569',
    'axes.labelcolor': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'axes.titlecolor': '#e2e8f0',
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'grid.color': '#2e3347',
    'savefig.facecolor': '#0f1117',
})
BRAND = '#6366f1'
TEAL = '#14b8a6'
ROSE = '#f43f5e'
YELLOW = '#eab308'

rng = np.random.default_rng(0)

## 1. Outcome vs trajectory

An agent reaches a goal via a *path*. We model each attempt as: it succeeds with prob `p`, and takes a number of steps (the oracle minimum is `opt`). Efficiency = opt / steps_taken.

In [ ]:
def attempt(p, opt, seed):
    g = np.random.default_rng(seed)
    solved = g.random() < p
    steps = opt + g.poisson(3)  # wandered a few extra steps
    return solved, steps

solved, steps = attempt(p=0.6, opt=5, seed=1)
print('solved:', solved, ' steps:', steps, ' efficiency:', round(5/steps, 2))

## 2. A single run lies — use pass@k and consistency

For a task an agent solves with probability `p` per attempt:
- **pass@k** = probability of solving in at least one of k tries = $1-(1-p)^k$.
- **consistency** (pass^k) = solving in *all* k tries = $p^k$.

A capable-but-flaky agent has high pass@k but low consistency.

In [ ]:
ks = np.arange(1, 11)
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for p in [0.3, 0.6]:
    ax.plot(ks, (1-(1-p)**ks)*100, 'o-', lw=2, label=f'pass@k (p={p})')
    ax.plot(ks, (p**ks)*100, 's--', lw=2, label=f'consistency p^k (p={p})')
ax.set_xlabel('k attempts'); ax.set_ylabel('%'); ax.set_title('pass@k rises, consistency falls')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 3. Estimate pass@k by Monte Carlo

Run many independent attempts and estimate the empirical pass@k — it should track $1-(1-p)^k$.

In [ ]:
def empirical_pass_at_k(p, k, trials=20000, seed=0):
    g = np.random.default_rng(seed)
    outcomes = g.random((trials, k)) < p
    return outcomes.any(axis=1).mean()

for k in [1, 3, 5]:
    emp = empirical_pass_at_k(0.4, k, seed=k)
    theo = 1-(1-0.4)**k
    print(f'k={k}: empirical {emp:.3f}  theory {theo:.3f}')

## ✏️ Your turn — pass@k

Implement `pass_at_k(p, k)` = probability of at least one success in `k` independent attempts.

In [ ]:
def pass_at_k(p, k):
    """TODO(you): return 1 - (1 - p)**k."""
    # TODO
    return ...


In [ ]:
print('pass@1 (p=0.4):', round(pass_at_k(0.4, 1), 3), '(expected 0.4)')
print('pass@5 (p=0.4):', round(pass_at_k(0.4, 5), 3), '(expected ~0.922)')
assert abs(pass_at_k(0.4, 1) - 0.4) < 1e-9
assert abs(pass_at_k(0.4, 5) - (1-0.6**5)) < 1e-9
assert abs(pass_at_k(0.0, 10) - 0.0) < 1e-9
assert abs(pass_at_k(1.0, 3) - 1.0) < 1e-9
print('\n✅ pass@k matches the closed form.')

<details>
<summary>Solution</summary>

```python
def pass_at_k(p, k):
    return 1 - (1 - p) ** k
```

Report pass@k *and* consistency: high pass@k with low consistency means capable-but-unreliable — not production-ready. And remember cost/latency budgets gate whether even a reliable agent ships.
</details>

## Recap

- Score **outcome** and **trajectory** (efficiency = opt/steps).
- A single stochastic run is noise; report **pass@k** (capability) and **consistency** (reliability).
- Monte-Carlo estimates track the closed form $1-(1-p)^k$.
- LLM-as-judge fills the gaps reference metrics can't — but validate the judge first.